# 11.3 只訓練 projector 能學到什麼？


已有視覺編碼器能區分紅與藍，文字模型也能從「red square」回答屬性，我們希望只調接頭把兩端聯繫起來。最簡單的任務是給紅方塊圖，目標描述寫red square。接頭因描述預測錯誤收到梯度，就有機會學哪些視覺數字應轉成文字模型可用的線索。

前置是[11.1尺寸與理解的區別](https://birdhackor.github.io/tiny-perceptron-vlm/11.1.html)、[11.2凍結名單](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)與[10.7展開後才錯位](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)。本例先驗證梯度路徑，仍用隨機模型，不能把一次反向計算叫成學會描述。真正能力訓練需先準備兩端已有的相關知識，否則接頭可能面對無法區分屬性的特徵或不懂目標的語言模型。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.data import ByteTokenizer
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, scene

torch.manual_seed(0)
tok = ByteTokenizer()
model = MultiModalLM(TinyLM(ModelConfig(width=8)))
model.requires_grad_(False)
model.image_projector.requires_grad_(True)
answer = tok.encode("red square") + [tok.eos_id]
prefix = [tok.bos_id, tok.user_id, tok.image_id, tok.eos_id, tok.assistant_id]
ids = torch.tensor(prefix + answer)
labels = torch.tensor([-100] * len(prefix) + answer)
out = model(ids, labels, image=scene())
loss = masked_loss(out["logits"], out["labels"])
loss.backward()
print("接頭收到梯度", model.image_projector.weight.grad.norm().item() > 0)
print("語言權重未累積梯度", model.language.embedding.weight.grad is None)

輸入包含紅方塊圖和完整未錯位序列。`ByteTokenizer`將英文字串逐位元組轉成編號，答案還加結束標記；prefix放開頭、用戶角色、圖片、用戶結束、助手角色。labels將前綴全部設忽略，只有回答與結束參與代價。模型內部把圖片標記展開16個位置，再執行下一位置對齊，`masked_loss`只在有效目標上算交叉熵。

`out["logits"]`是每個輸入位置對全部詞表候選的原始分數，稱為[logits](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)。[交叉熵](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)先把這組分數轉成候選機率，再用正確目標編號衡量猜錯代價；正確候選機率越低，代價越高。`masked_loss`依`out["labels"]`排除-100位置，將其餘位置的代價平均，得到單個數字`loss`，再由它求出梯度。

兩行正常均為True：接頭收到非零梯度，凍結的文字embedding沒有自己的梯度。計算信號能夠穿過固定文字模型回到接頭，正是只訓接頭所需的路徑。程式沒有優化器更新，也沒有新圖測試，因此尚未得到描述成功率。

只訓接頭常用於先對齊描述；之後遇到問答可能需要進一步調整語言層。

red square共有10個英文字母與空白，都屬於ASCII這個單位元組字元集合，所以這個位元組切詞也有10個正文編號，加結束一共11個有效目標。使用者和圖片位置沒有被當作要生成的文字；它們仍影響這11次預測。若標籤把圖片位置也設成普通文字答案，就等於要求模型在沒有相應文字真值的格子猜字，改變了原任務。只訓接頭可以重新組合固定特徵，卻不能把編碼器本來丟掉的資訊找回來；因此「基模會用相關文字線索」與「編碼器能分相關屬性」是能力實驗的前提，不是可省略的背景。

練習把`image_projector.requires_grad_(True)`改False，並在`loss.backward()`前加上`print(sum(p.numel() for p in model.parameters() if p.requires_grad))`，沿用[11.2](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)的方法數出可訓練參數總數。先預測它印0，接著`backward()`會報錯，因為代價已沒有連到任何需計算梯度的參數，再執行核對。這是故意關閉所有學習路徑的反例；合理做法是在啟動訓練前拒絕這個配置，不是讓它空跑許多步後期待進步。
